# BERTopic Exploration - Enhanced Version

This notebook performs topic modeling on fishing survey data with improved:
- Data preprocessing and validation
- Modular helper functions
- Better error handling
- Model persistence for reproducibility

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
from bertopic import BERTopic
import matplotlib.pyplot as plt
import seaborn as sns
import random
import os
import re
from typing import List, Tuple, Dict

# Set seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# Set PyTorch seed if available
try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(SEED)
        torch.cuda.manual_seed_all(SEED)
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(SEED)
    # Set deterministic behavior
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
except ImportError:
    pass

# Set display options for better readability
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

print(f"✅ Random seed set to {SEED} for reproducibility")
print("All random number generators have been seeded.")

## Helper Functions

These functions encapsulate reusable logic for data preprocessing, model setup, and result display.

In [ ]:
# ============================================================================
# DATA PREPROCESSING HELPERS
# ============================================================================

def clean_text(text: str) -> str:
    """Clean individual text response.
    
    Args:
        text: Raw text string
        
    Returns:
        Cleaned text string
    """
    # Convert to string and strip whitespace
    text = str(text).strip()
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text)
    
    return text


def preprocess_responses(responses: List[str], 
                        min_length: int = 2,
                        remove_duplicates: bool = False,
                        verbose: bool = True) -> Tuple[List[str], Dict]:
    """Preprocess survey responses with cleaning and filtering.
    
    Args:
        responses: List of raw text responses
        min_length: Minimum word count for a response to be kept
        remove_duplicates: Whether to remove exact duplicate responses
        verbose: Whether to print statistics
        
    Returns:
        Tuple of (cleaned_responses, stats_dict)
    """
    original_count = len(responses)
    stats = {'original_count': original_count}
    
    # Clean all responses
    cleaned = [clean_text(r) for r in responses]
    
    # Remove empty responses
    cleaned = [r for r in cleaned if r and r.lower() not in ['', 'nan', 'none', 'n/a']]
    stats['after_empty_removal'] = len(cleaned)
    stats['empty_removed'] = original_count - len(cleaned)
    
    # Filter by minimum length
    length_filtered = [r for r in cleaned if len(r.split()) >= min_length]
    stats['after_length_filter'] = len(length_filtered)
    stats['short_removed'] = len(cleaned) - len(length_filtered)
    
    # Remove duplicates if requested
    if remove_duplicates:
        before_dedup = len(length_filtered)
        length_filtered = list(dict.fromkeys(length_filtered))  # Preserves order
        stats['after_deduplication'] = len(length_filtered)
        stats['duplicates_removed'] = before_dedup - len(length_filtered)
    
    stats['final_count'] = len(length_filtered)
    
    if verbose:
        print("=" * 60)
        print("DATA PREPROCESSING SUMMARY")
        print("=" * 60)
        print(f"Original responses: {stats['original_count']}")
        print(f"Empty/invalid removed: {stats['empty_removed']}")
        print(f"Too short (< {min_length} words) removed: {stats['short_removed']}")
        if remove_duplicates:
            print(f"Duplicates removed: {stats['duplicates_removed']}")
        print(f"\n✅ Final response count: {stats['final_count']}")
        print(f"   Retention rate: {stats['final_count']/stats['original_count']*100:.1f}%")
        print("=" * 60)
    
    return length_filtered, stats


def analyze_response_statistics(responses: List[str]) -> None:
    """Print detailed statistics about the responses.
    
    Args:
        responses: List of text responses
    """
    word_counts = [len(r.split()) for r in responses]
    char_counts = [len(r) for r in responses]
    
    print("\n📊 Response Statistics:")
    print(f"   Total responses: {len(responses)}")
    print(f"\n   Word count per response:")
    print(f"      Mean: {np.mean(word_counts):.1f}")
    print(f"      Median: {np.median(word_counts):.0f}")
    print(f"      Min: {np.min(word_counts)}")
    print(f"      Max: {np.max(word_counts)}")
    print(f"\n   Character count per response:")
    print(f"      Mean: {np.mean(char_counts):.1f}")
    print(f"      Median: {np.median(char_counts):.0f}")
    
    # Check for duplicates
    duplicates = pd.Series(responses).value_counts()
    if (duplicates > 1).any():
        print(f"\n   ⚠️ Found {(duplicates > 1).sum()} duplicate responses:")
        for resp, count in duplicates[duplicates > 1].head(5).items():
            print(f"      '{resp[:50]}...' appears {count} times")
    else:
        print(f"\n   ✅ No duplicate responses found")


# ============================================================================
# MODEL SETUP HELPERS
# ============================================================================

def setup_device() -> str:
    """Detect and configure the best available device for computation.
    
    Returns:
        Device string: 'mps', 'cuda', or 'cpu'
    """
    try:
        import torch
        if torch.backends.mps.is_available():
            print("✅ Using Apple Silicon GPU (Metal) for acceleration")
            return "mps"
        elif torch.cuda.is_available():
            print("✅ Using NVIDIA CUDA GPU")
            return "cuda"
    except ImportError:
        pass
    
    print("⚠️ Using CPU (no GPU detected)")
    return "cpu"


def create_bertopic_model(seed: int = 42,
                         min_cluster_size: int = 3,
                         umap_n_neighbors: int = 15,
                         umap_n_components: int = 5) -> BERTopic:
    """Create and configure a BERTopic model with specified parameters.
    
    Args:
        seed: Random seed for reproducibility
        min_cluster_size: Minimum size for HDBSCAN clusters
        umap_n_neighbors: Number of neighbors for UMAP
        umap_n_components: Number of UMAP dimensions
        
    Returns:
        Configured BERTopic model instance
    """
    from sklearn.feature_extraction.text import CountVectorizer
    from sentence_transformers import SentenceTransformer
    from umap import UMAP
    from hdbscan import HDBSCAN
    
    # Set seed for embedding model
    embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
    
    # Create UMAP model with seed for reproducibility
    umap_model = UMAP(
        n_neighbors=umap_n_neighbors,
        n_components=umap_n_components,
        min_dist=0.0,
        metric='cosine',
        random_state=seed
    )
    
    # Create HDBSCAN model with deterministic parameters
    hdbscan_model = HDBSCAN(
        min_cluster_size=min_cluster_size,
        metric='euclidean',
        cluster_selection_method='eom',
        prediction_data=True
    )
    
    # Create vectorizer
    vectorizer_model = CountVectorizer(stop_words="english", ngram_range=(1, 2))
    
    # Create BERTopic model with all seeded components
    topic_model = BERTopic(
        language="english",
        calculate_probabilities=True,
        verbose=True,
        embedding_model=embedding_model,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        nr_topics=None
    )
    
    print(f"\n✅ BERTopic model created with:")
    print(f"   - min_cluster_size: {min_cluster_size}")
    print(f"   - umap_n_neighbors: {umap_n_neighbors}")
    print(f"   - umap_n_components: {umap_n_components}")
    
    return topic_model


def setup_representation_model(api_key: str = None, seed: int = 42):
    """Create appropriate representation model (OpenAI or local).
    
    Args:
        api_key: OpenAI API key (optional)
        seed: Random seed for local models
        
    Returns:
        Configured representation model
    """
    device = setup_device()
    
    if api_key:
        print("✅ Using GPT-4o mini for topic labeling")
        
        from openai import OpenAI as OpenAIClient
        from bertopic.representation import OpenAI
        
        client = OpenAIClient(api_key=api_key)
        
        representation_model = OpenAI(
            client=client,
            model="gpt-4o-mini",
            chat=True,
            nr_docs=10,
            diversity=0.5,
            delay_in_seconds=2,
            exponential_backoff=True,
            prompt="""I have a topic that is part of a survey about fishing site preferences. 

The topic contains these keywords: [KEYWORDS]

Representative survey responses from this topic:
[DOCUMENTS]

Your task: Create a concise, distinctive label that captures what makes THIS specific topic unique and different from other fishing-related topics.

Important guidelines:
- DO NOT use generic labels like "Fishing Catch" or "Fishing Preferences" - be SPECIFIC to what distinguishes this topic
- Focus on the PRIMARY concern or theme that unites these responses
- Use natural, human-readable language that someone would actually say
- Be concise but descriptive enough to differentiate this topic from others
- Look at the actual survey responses to understand the nuance

Generate ONLY the label - nothing else:"""
        )
        
        print("⚠️ Note: OpenAI API responses may vary slightly even with the same inputs.")
        
    else:
        print("⚠️ Using local Flan-T5-large model (no API key required)")
        
        from transformers import pipeline, set_seed
        from bertopic.representation import TextGeneration
        
        set_seed(seed)
        
        pipeline_device = 0 if device in ['mps', 'cuda'] else -1
        if device == 'mps':
            os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
        
        generator = pipeline(
            'text2text-generation',
            model='google/flan-t5-large',
            max_length=50,
            device=pipeline_device
        )
        
        prompt = """Topic keywords: [KEYWORDS]
Example responses: [DOCUMENTS]

Create a SPECIFIC, DISTINCTIVE label for this fishing survey topic.
Focus on what makes THIS topic unique - not just "fishing" or "catch".
Look at the actual responses to understand the specific concern.

Label:"""
        
        representation_model = TextGeneration(generator, prompt=prompt, nr_docs=10, diversity=0.5)
        print("✅ Local model is fully deterministic with the seed set.")
    
    return representation_model


# ============================================================================
# DISPLAY AND ANALYSIS HELPERS
# ============================================================================

def display_topic_summary(topic_model, topics: List[int], responses: List[str], 
                         top_n: int = 10) -> None:
    """Display formatted summary of topics with examples.
    
    Args:
        topic_model: Fitted BERTopic model
        topics: List of topic assignments
        responses: List of text responses
        top_n: Number of top topics to display
    """
    topic_info = topic_model.get_topic_info()
    topic_info = topic_info[topic_info['Topic'] != -1].sort_values('Count', ascending=False)
    
    print("=" * 80)
    print("TOP TOPICS WITH DISTINCTIVE LABELS")
    print("=" * 80)
    
    for idx, (_, row) in enumerate(topic_info.head(top_n).iterrows(), 1):
        topic_id = row['Topic']
        count = row['Count']
        percentage = (count / len(responses)) * 100
        
        print(f"\n📌 Rank #{idx} | {row['Name']} ({count} responses, {percentage:.1f}%)")
        
        # Get keywords
        keywords = [word for word, _ in topic_model.get_topic(topic_id)[:8]]
        print(f"   Keywords: {', '.join(keywords)}")
        
        # Get example documents
        topic_docs = [responses[i] for i, t in enumerate(topics) if t == topic_id][:3]
        print(f"   Examples:")
        for doc in topic_docs:
            print(f"     • {doc[:90]}{'...' if len(doc) > 90 else ''}")
        print("-" * 80)


def print_model_statistics(topics: List[int], responses: List[str]) -> None:
    """Print statistics about the topic modeling results.
    
    Args:
        topics: List of topic assignments
        responses: List of text responses
    """
    n_topics = len(set(topics)) - 1  # Exclude outlier topic
    n_outliers = sum(1 for t in topics if t == -1)
    outlier_pct = (n_outliers / len(responses)) * 100
    
    print("\n" + "=" * 60)
    print("MODEL RESULTS SUMMARY")
    print("=" * 60)
    print(f"Total responses: {len(responses)}")
    print(f"Topics identified: {n_topics}")
    print(f"Outliers: {n_outliers} ({outlier_pct:.1f}%)")
    
    if outlier_pct > 20:
        print(f"\n⚠️ Warning: High outlier rate ({outlier_pct:.1f}%)")
        print("   Consider adjusting min_cluster_size or UMAP parameters")
    elif outlier_pct < 5:
        print(f"\n✅ Excellent: Low outlier rate ({outlier_pct:.1f}%)")
    else:
        print(f"\n✅ Good: Acceptable outlier rate ({outlier_pct:.1f}%)")
    
    print("=" * 60)


def save_results(topic_model, topics: List[int], responses: List[str], 
                model_name: str = "bertopic_model") -> None:
    """Save model and results to files.
    
    Args:
        topic_model: Fitted BERTopic model
        topics: List of topic assignments
        responses: List of text responses
        model_name: Base name for saved files
    """
    # Save the model
    model_path = f"{model_name}_{SEED}"
    topic_model.save(model_path, serialization="safetensors", save_ctfidf=True)
    print(f"✅ Model saved to: {model_path}")
    
    # Save topic assignments
    topic_info = topic_model.get_topic_info()
    topic_labels = {}
    for _, row in topic_info.iterrows():
        topic_labels[row['Topic']] = row['Name']
    
    results_df = pd.DataFrame({
        'response': responses,
        'topic_id': topics,
        'topic_label': [topic_labels.get(t, 'Outlier') for t in topics]
    })
    results_df.to_csv(f'{model_name}_assignments.csv', index=False)
    print(f"✅ Topic assignments saved to: {model_name}_assignments.csv")
    
    # Save topic summary
    topic_info.to_csv(f'{model_name}_summary.csv', index=False)
    print(f"✅ Topic summary saved to: {model_name}_summary.csv")


print("✅ All helper functions loaded successfully!")

## Data Loading and Validation

In [ ]:
# Load the survey data with error handling
try:
    df = pd.read_csv('test_survey.csv')
    
    if df.shape[1] < 1:
        raise ValueError("CSV must have at least one column")
    
    print(f"✅ Dataset loaded successfully")
    print(f"   Shape: {df.shape}")
    print(f"\n   Column name: {df.columns[0]}")
    
except FileNotFoundError:
    print("❌ Error: test_survey.csv not found in current directory")
    raise
except Exception as e:
    print(f"❌ Error loading CSV: {e}")
    raise

# Display first few rows
print("\nFirst few rows:")
df.head()

## Data Preprocessing

Clean and filter responses to improve topic modeling quality.

In [ ]:
# Extract responses from the dataframe
raw_responses = df.iloc[:, 0].dropna().astype(str).tolist()

# Remove the question header (first row)
raw_responses = raw_responses[1:]

print(f"Raw responses extracted: {len(raw_responses)}\n")

# Analyze raw data before preprocessing
analyze_response_statistics(raw_responses)

In [ ]:
# Preprocess responses
responses, preprocessing_stats = preprocess_responses(
    raw_responses,
    min_length=2,  # Keep responses with at least 2 words
    remove_duplicates=False,  # Set to True to remove exact duplicates
    verbose=True
)

# Show statistics after preprocessing
analyze_response_statistics(responses)

In [ ]:
# Display sample of processed responses
print("\n📝 Sample of processed responses:")
for i, resp in enumerate(responses[:10], 1):
    print(f"{i:2d}. {resp}")

## Model Training

In [ ]:
# Create BERTopic model with configurable parameters
topic_model = create_bertopic_model(
    seed=SEED,
    min_cluster_size=3,  # Adjust this to control granularity
    umap_n_neighbors=15,
    umap_n_components=5
)

# Fit the model
print("\n🔄 Training BERTopic model...\n")
topics, probs = topic_model.fit_transform(responses)

# Print statistics
print_model_statistics(topics, responses)

In [ ]:
# View initial topics with keywords
topic_info = topic_model.get_topic_info()

print("Initial Topics (by keyword extraction):")
print(topic_info[['Topic', 'Count', 'Name']].to_string())

print("\n" + "=" * 60)
print("Topic distribution:")
print(pd.Series(topics).value_counts().sort_index())

## LLM-Enhanced Topic Labeling

In [ ]:
# Set up OpenAI API key (optional)
import getpass

print("OpenAI API Key Setup (Optional)")
print("=" * 50)
print("Press Enter to skip and use local model instead.\n")

api_key = None

if 'OPENAI_API_KEY' in os.environ and os.environ['OPENAI_API_KEY']:
    print("✅ API key found in environment")
    api_key = os.environ['OPENAI_API_KEY']
else:
    user_input = getpass.getpass("Enter OpenAI API key (or press Enter to skip): ")
    if user_input.strip():
        api_key = user_input.strip()
        os.environ['OPENAI_API_KEY'] = api_key
        print("✅ API key set")
    else:
        print("⏭️ Skipping - will use local model")

In [ ]:
# Generate distinctive topic labels using LLM
representation_model = setup_representation_model(api_key=api_key, seed=SEED)

# Estimate cost if using OpenAI
if api_key:
    num_topics = len(set(topics)) - 1
    estimated_calls = num_topics + 1  # Topics + outliers
    estimated_cost = estimated_calls * 0.00015  # Rough GPT-4o-mini cost per call
    print(f"\n💰 Estimated API calls: ~{estimated_calls}")
    print(f"   Estimated cost: ${estimated_cost:.4f}")

print("\n🔄 Generating distinctive topic labels... This may take a minute.\n")
topic_model.update_topics(responses, representation_model=representation_model)

print("\n✅ Topic labels generated successfully!")

## Results Analysis

In [ ]:
# Display topic summary with updated labels
display_topic_summary(topic_model, topics, responses, top_n=10)

In [ ]:
# Show all topics in a table
topic_info = topic_model.get_topic_info()
print("\nAll Topics Summary:")
print(topic_info[['Topic', 'Count', 'Name']].to_string())

## Visualizations

In [ ]:
# Bar chart of top topics
fig_bar = topic_model.visualize_barchart(top_n_topics=10, height=400)
fig_bar.show()

# Save visualization
fig_bar.write_html("topic_barchart.html")
print("\n✅ Bar chart saved to: topic_barchart.html")

In [ ]:
# Intertopic distance map
fig_map = topic_model.visualize_topics()
fig_map.show()

# Save visualization
fig_map.write_html("topic_distance_map.html")
print("\n✅ Distance map saved to: topic_distance_map.html")

## Hierarchical Topic Structure

In [ ]:
# Build hierarchical topics
print("🔄 Building hierarchical topic structure...\n")
hierarchical_topics = topic_model.hierarchical_topics(responses)

print(f"✅ Hierarchy created with {len(hierarchical_topics)} merges")

In [ ]:
# Visualize hierarchy
fig_hierarchy = topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)
fig_hierarchy.show()

# Save visualization
fig_hierarchy.write_html("topic_hierarchy.html")
print("\n✅ Hierarchy visualization saved to: topic_hierarchy.html")

In [ ]:
# Show hierarchical merging structure
hierarchy_df = hierarchical_topics.copy().sort_values('Distance')

print("\nHierarchical Merging Structure:")
print("(Topics merge together as you move up the hierarchy)\n")
print(hierarchy_df[['Parent_ID', 'Topics', 'Distance']].head(20).to_string())

print("\nNote: Lower distances = more similar topics that merge first")

## Save Results

In [ ]:
# Save model and results
save_results(
    topic_model=topic_model,
    topics=topics,
    responses=responses,
    model_name="fishing_survey_bertopic"
)

print("\n✅ All results saved! You can load the model later with:")
print(f"   topic_model = BERTopic.load('fishing_survey_bertopic_{SEED}')")